# Wav to Spectrogram

Part of code is utilized from https://dev.to/dolbyio/a-beginners-guide-to-visualizing-audio-as-a-spectrogram-in-python-1mm8?comments_sort=top

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io.wavfile as wavfile
from scipy import signal
from pathlib import Path
import os
import librosa

In [2]:
def change_gain(Fs, aud, db=0):
    aud = aud.astype(np.float32)
    if aud.ndim > 1:
        aud = aud.mean(axis=1)
    aud = aud / (np.max(np.abs(aud)) + 1e-9)
    aud = aud * 10 ** (db / 20)
    return np.clip(aud, -1, 1)

def change_speed(Fs, aud, rate=1.0):
    aud = aud.astype(np.float32)
    if aud.ndim > 1:
            aud = aud.mean(axis=1)
    return librosa.effects.time_stretch(aud, rate=rate)

def convert(path, output_folder, output_filename, output_chordname, rate=1.0, db=0):
    Fs, aud = wavfile.read(path)
    aud = change_speed(Fs, aud, rate)
    aud = change_gain(Fs, aud, db)
    powerSpectrum, frequenciesFound, time, imageAxis = plt.specgram(aud, Fs=Fs)
    output_path = os.path.join(output_folder, f"{output_filename} {output_chordname}.png")
    plt.axis("off")
    plt.savefig(output_path, bbox_inches="tight", pad_inches=0)
    plt.close()
    print(f"Spectrogram saved: {output_path}")

In [3]:
# Directory where the chord dataset is stored
XC_ROOTDIR = '../data/Training Data/'
XC_DIR = 'wavs'

# Initialize an empty list to store file details
filelist = []

chords_dir = os.path.join(os.getcwd(), XC_ROOTDIR, XC_DIR)
for chord_folder in os.listdir(chords_dir):
    chords_path = os.path.join(chords_dir, chord_folder)
    if os.path.isdir(chords_path):  # Ensure it's a directory
        for file_name in os.listdir(chords_path):
            if file_name.endswith('.wav'):  # Filter for .wav files
                filelist.append({
                    'fullfilename': os.path.join(chords_path, file_name),
                    'filename': Path(file_name).stem,
                    'chord': chord_folder  # The folder name is the chord
                })

# Create a DataFrame with the gathered file details
df_files_wav = pd.DataFrame(filelist)

# Display a summary of the DataFrame
print('=====================================================')
print(f'Number of files: {len(df_files_wav)}')
print(f'Number of chords: {len(df_files_wav["chord"].unique())}')
print('=====================================================')

# Print the first few rows of the DataFrame to verify
df_files_wav

Number of files: 475
Number of chords: 8


,fullfilename,filename,chord
0,/Users/marktringas/chordmaxxer/wav-to-spectrog...,Nimbusparken 40 94,Am
1,/Users/marktringas/chordmaxxer/wav-to-spectrog...,ArgSG_Am_open_122_ID4_1,Am
2,/Users/marktringas/chordmaxxer/wav-to-spectrog...,Nimbusparken 40 95,Am
3,/Users/marktringas/chordmaxxer/wav-to-spectrog...,Nimbusparken 40 97,Am
4,/Users/marktringas/chordmaxxer/wav-to-spectrog...,ArgSG_Am_open_122_ID4_2,Am
...,...,...,...
470,/Users/marktringas/chordmaxxer/wav-to-spectrog...,ArgSG_E_open_12020_ID4_4,E
471,/Users/marktringas/chordmaxxer/wav-to-spectrog...,ArgSG_E_open_022_ID4_3,E
472,/Users/marktringas/chordmaxxer/wav-to-spectrog...,Nimbusparken 40 140,E
473,/Users/marktringas/chordmaxxer/wav-to-spectrog...,ArgSG_E_open_022_ID4_2,E


In [6]:
def process_files(dataframe, output_folder):
    counter = 1
    for _, row in dataframe.iterrows():
        input_wav_file = row['fullfilename']
        filename = row['filename']
        chord_name = row['chord']
        
        # Create chord folder
        chord_folder = os.path.join(output_folder, chord_name)
        os.makedirs(chord_folder, exist_ok=True)
    
        print(f"\nProcessing {counter}/{len(dataframe)}, {input_wav_file} for chord: {chord_name}")
        convert(input_wav_file, chord_folder, filename, chord_name)
        convert(input_wav_file, chord_folder, filename + "0.9", chord_name, rate=0.9)
        convert(input_wav_file, chord_folder, filename + "0.95", chord_name, rate=0.95)
        convert(input_wav_file, chord_folder, filename + "1.05", chord_name, rate=1.05)
        convert(input_wav_file, chord_folder, filename + "1.1", chord_name, rate=1.1)
        convert(input_wav_file, chord_folder, filename + "-0.5", chord_name, db=-0.5)
        convert(input_wav_file, chord_folder, filename + "-0.25", chord_name, db=-0.25)
        convert(input_wav_file, chord_folder, filename + "0.25", chord_name, db=0.25)
        convert(input_wav_file, chord_folder, filename + "0.5", chord_name, db=0.5)

        counter += 1
    
    print("\n\nAll files have been processed.")

output_root_folder = '../data/Training Data/pngs/'

In [7]:
process_files(df_files_wav, output_root_folder)


Processing 1/475, /Users/marktringas/chordmaxxer/wav-to-spectrogram/../data/Training Data/wavs/Am/Nimbusparken 40 94.wav for chord: Am
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 94 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 940.9 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 940.95 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 941.05 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 941.1 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 94-0.5 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 94-0.25 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 940.25 Am.png
Spectrogram saved: ../data/Training Data/pngs/Am/Nimbusparken 40 940.5 Am.png

Processing 2/475, /Users/marktringas/chordmaxxer/wav-to-spectrogram/../data/Training Data/wavs/Am/ArgSG_Am_open_122_ID4_1.wav for chord: Am
Spectrogram saved: